# MedTrack_DV — Module 1: Hospital Data Collection

**Milestone 1, Week 1–2**

Goal: collect and integrate the hospital / patient admission data into a single
raw dataset (`hospital_raw_data.csv`) that later modules (cleaning, KPI
engineering, dashboards) will build on.

Tasks covered in this notebook:
1. Download / load hospital operational dataset
2. Collect patient admission records
3. Gather department and resource data
4. Integrate into one dataset
5. Check completeness (target: >95%)


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

## 1. Load the source dataset

In [2]:
raw = pd.read_csv("hospital_data_analysis.csv")
print(f"Rows: {len(raw)}  |  Columns: {len(raw.columns)}")
raw.head()

Rows: 984  |  Columns: 10


,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Patient_ID      984 non-null    int64
 1   Age             984 non-null    int64
 2   Gender          984 non-null    str  
 3   Condition       984 non-null    str  
 4   Procedure       984 non-null    str  
 5   Cost            984 non-null    int64
 6   Length_of_Stay  984 non-null    int64
 7   Readmission     984 non-null    str  
 8   Outcome         984 non-null    str  
 9   Satisfaction    984 non-null    int64
dtypes: int64(5), str(5)
memory usage: 77.0 KB


## 2. Add Department (from Condition)

The source file is patient/clinical-level and doesn't include hospital
operations fields (Department, Hospital, Region, dates). We derive
**Department** from **Condition** using a documented mapping so later
modules (Department Analytics, Resource Utilization) have something to
group on.

In [4]:
CONDITION_TO_DEPARTMENT = {
    "Heart Disease": "Cardiology",
    "Heart Attack": "Cardiology",
    "Diabetes": "General Medicine",
    "Hypertension": "General Medicine",
    "Respiratory Infection": "General Medicine",
    "Kidney Stones": "General Medicine",
    "Osteoarthritis": "Orthopedics",
    "Fractured Arm": "Orthopedics",
    "Fractured Leg": "Orthopedics",
    "Stroke": "ICU",
    "Cancer": "Surgery",
    "Prostate Cancer": "Surgery",
    "Appendicitis": "Surgery",
    "Allergic Reaction": "Emergency",
    "Childbirth": "Pediatrics",
}

raw["Department"] = raw["Condition"].map(CONDITION_TO_DEPARTMENT)
raw["Department"] = raw["Department"].fillna("General Medicine")
raw["Department"].value_counts()

Department
General Medicine    261
Orthopedics         197
Surgery             197
Cardiology          132
ICU                  66
Emergency            66
Pediatrics           65
Name: count, dtype: int64

## 3. Add Hospital, Region, Patient Type, and Admission/Discharge dates

These are generated with a fixed random seed (reproducible) to simulate the
operational fields a real hospital system would provide, using the same
5 hospitals / 5 regions referenced in the project brief.

In [5]:
HOSPITALS = ["City Care Hospital", "Green Valley Hospital", "Sunrise Medical Center",
             "Metro Health Institute", "HealthPlus Hospital"]
REGIONS = ["North", "South", "East", "West", "Central"]
PATIENT_TYPES = ["Inpatient", "Outpatient", "Emergency", "Day Care"]
PATIENT_TYPE_WEIGHTS = [0.58, 0.32, 0.07, 0.03]

raw["Hospital"] = rng.choice(HOSPITALS, size=len(raw))
raw["Region"] = rng.choice(REGIONS, size=len(raw))
raw["Patient_Type"] = rng.choice(PATIENT_TYPES, size=len(raw), p=PATIENT_TYPE_WEIGHTS)

start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-12-31")
offsets = rng.integers(0, (end - start).days + 1, size=len(raw))
raw["Admission_Date"] = start + pd.to_timedelta(offsets, unit="D")
raw["Discharge_Date"] = raw["Admission_Date"] + pd.to_timedelta(raw["Length_of_Stay"], unit="D")

raw[["Hospital", "Region", "Patient_Type", "Admission_Date", "Discharge_Date"]].head()

,Hospital,Region,Patient_Type,Admission_Date,Discharge_Date
0,City Care Hospital,East,Inpatient,2024-07-10,2024-07-15
1,Metro Health Institute,West,Emergency,2024-06-21,2024-06-24
2,Metro Health Institute,West,Inpatient,2024-09-21,2024-09-22
3,Sunrise Medical Center,South,Inpatient,2024-07-05,2024-07-12
4,Sunrise Medical Center,South,Inpatient,2024-07-18,2024-07-28


## 4. Reorder columns and integrate into one dataset

In [6]:
column_order = [
    "Patient_ID", "Hospital", "Region", "Department", "Patient_Type",
    "Admission_Date", "Discharge_Date", "Age", "Gender", "Condition",
    "Procedure", "Cost", "Length_of_Stay", "Readmission", "Outcome",
    "Satisfaction",
]
hospital_data = raw[column_order]
hospital_data.head()

,Patient_ID,Hospital,Region,Department,Patient_Type,Admission_Date,Discharge_Date,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,City Care Hospital,East,Cardiology,Inpatient,2024-07-10,2024-07-15,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,Metro Health Institute,West,General Medicine,Emergency,2024-06-21,2024-06-24,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,Metro Health Institute,West,Orthopedics,Inpatient,2024-09-21,2024-09-22,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,Sunrise Medical Center,South,ICU,Inpatient,2024-07-05,2024-07-12,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,Sunrise Medical Center,South,Surgery,Inpatient,2024-07-18,2024-07-28,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


## 5. Data quality check — dataset completeness (target: >95%)

In [7]:
missing_by_col = hospital_data.isna().sum()
completeness = 100 * (1 - missing_by_col.sum() / hospital_data.size)

print(missing_by_col)
print(f"\nOverall dataset completeness: {completeness:.2f}%")
assert completeness > 95, "Completeness target not met"
print("Completeness target met ✔")

Patient_ID        0
Hospital          0
Region            0
Department        0
Patient_Type      0
Admission_Date    0
Discharge_Date    0
Age               0
Gender            0
Condition         0
Procedure         0
Cost              0
Length_of_Stay    0
Readmission       0
Outcome           0
Satisfaction      0
dtype: int64

Overall dataset completeness: 100.00%
Completeness target met ✔


## 6. Save integrated raw dataset

In [8]:
hospital_data.to_csv("hospital_raw_data.csv", index=False)
print(f"Saved hospital_raw_data.csv  |  {len(hospital_data)} rows, {len(hospital_data.columns)} columns")

Saved hospital_raw_data.csv  |  984 rows, 16 columns


## Summary

| Item | Value |
|---|---|
| Source file | `hospital_data_analysis.csv` |
| Output file | `hospital_raw_data.csv` |
| Rows | 984 |
| Columns | 16 |
| Hospitals | 5 |
| Regions | 5 |
| Departments | 7 |
| Completeness | 100% |

**Next:** Module 2 — Data Cleaning & Transformation (`hospital_cleaning.ipynb`).